# Python Import and Zipimport: Step-by-Step

这个 notebook 用互动方式解释：为什么普通 `.py` 文件可以被 `import`，为什么 `.zip` 文件也可以被 `import`，以及 `PYTHONPATH` 和 `sys.path` 到底是什么。

请从上到下逐格运行。每一节都依赖前面 cell 创建的变量。示例默认运行在 Linux/Ubuntu 环境。

## 0. 学习目标

运行完这个 notebook 后，你应该能解释：

- Python 执行 `import name` 时会去哪里找 `name`。
- `sys.path` 是运行时的 import 搜索路径列表。
- `PYTHONPATH` 是 Python 启动前读入的环境变量，会追加到 `sys.path`。
- `.zip` 文件为什么可以像目录一样被 Python import。
- flat module zip 和 package zip 的结构差异。
- 为什么从项目根目录运行脚本通常不需要额外配置 import 路径。

In [ ]:
import importlib
import os
import subprocess
import sys
import tempfile
import zipfile
import zipimport
from pathlib import Path

print(sys.version)
print("current working directory:", Path.cwd())
print("\nfirst sys.path entries:")
for idx, entry in enumerate(sys.path[:8]):
    print(f"{idx}: {entry!r}")

## 1. 先看普通目录 import

Python 不会扫描整个磁盘。它只会在 `sys.path` 列表里的位置查找模块。

下面我们创建一个临时目录，并在里面写一个普通的 `hello_module.py`。

In [ ]:
lab = Path(tempfile.mkdtemp(prefix="zipimport_lab_"))
normal_dir = lab / "normal_modules"
normal_dir.mkdir()

module_file = normal_dir / "hello_module.py"
module_file.write_text(
    "VALUE = 41\n\n"
    "def answer():\n"
    "    return VALUE + 1\n",
    encoding="utf-8",
)

print("lab directory:", lab)
print("created:", module_file)
print(module_file.read_text(encoding="utf-8"))

现在这个目录还不一定在 `sys.path` 里，所以直接 import 可能失败。我们把目录插入到 `sys.path[0]`，再 import。

In [ ]:
sys.modules.pop("hello_module", None)
if str(normal_dir) not in sys.path:
    sys.path.insert(0, str(normal_dir))

import hello_module

print("hello_module.__file__:", hello_module.__file__)
print("hello_module.answer():", hello_module.answer())
assert hello_module.answer() == 42

关键点：`import hello_module` 能成功，不是因为 Python 知道你的临时目录，而是因为我们把这个目录加入了 `sys.path`。

从项目根目录运行脚本时，Python 会自动把当前工作目录放进 `sys.path`，所以根目录下的 `.py` 文件可以互相 import。

## 2. 创建一个 flat-module zip

Python 标准库支持从 zip 文件 import `.py` 模块。这个机制叫 `zipimport`。

先创建一个 zip，里面直接放两个模块：

```text
flat_modules.zip
  alpha.py
  beta.py
```

`beta.py` 会 import `alpha.py`。

In [ ]:
flat_zip = lab / "flat_modules.zip"
with zipfile.ZipFile(flat_zip, "w") as zf:
    zf.writestr(
        "alpha.py",
        "NAME = 'alpha from zip'\n\n"
        "def value():\n"
        "    return 10\n",
    )
    zf.writestr(
        "beta.py",
        "from alpha import value\n\n"
        "def total():\n"
        "    return value() + 5\n",
    )

print("created:", flat_zip)
with zipfile.ZipFile(flat_zip) as zf:
    print("zip contents:", zf.namelist())

现在把 zip 文件本身加入 `sys.path`。注意：加入的是 zip 文件路径，不是解压后的目录。

In [ ]:
for name in ["alpha", "beta"]:
    sys.modules.pop(name, None)

if str(flat_zip) not in sys.path:
    sys.path.insert(0, str(flat_zip))
importlib.invalidate_caches()

import beta

print("beta.__file__:", beta.__file__)
print("beta.total():", beta.total())
assert beta.total() == 15

`beta.__file__` 通常会显示类似：

```text
/tmp/.../flat_modules.zip/beta.py
```

这说明 Python 没有解压 zip，而是直接从 zip 里读取 `beta.py`。`beta.py` 里的 `from alpha import value` 也能工作，因为 `alpha.py` 和 `beta.py` 在 zip 根目录中是同级模块。

## 3. `sys.path` 和 `PYTHONPATH` 的区别

`sys.path` 是 Python 进程运行时实际使用的列表。你可以在代码中修改它。

`PYTHONPATH` 是启动 Python 前设置的环境变量。Python 启动时会读取它，然后把里面的路径加入 `sys.path`。

下面用一个子进程证明：只设置 `PYTHONPATH`，子进程就能 import zip 里的 `beta`。

In [ ]:
code = "import beta; print(beta.__file__); print(beta.total())"
env = os.environ.copy()
env["PYTHONPATH"] = str(flat_zip)

result = subprocess.run(
    [sys.executable, "-c", code],
    env=env,
    text=True,
    capture_output=True,
    check=True,
)

print(result.stdout)

Ubuntu shell 里手动设置 `PYTHONPATH` 的形式是：

```bash
PYTHONPATH=/path/to/flat_modules.zip python -c "import beta; print(beta.total())"
```

这和在 Python 代码里执行 `sys.path.insert(0, '/path/to/flat_modules.zip')` 的效果相近，只是发生的时机不同。

## 4. package 结构也可以放进 zip

除了 flat module，zip 里也可以放 package：

```text
package_modules.zip
  demo_pkg/
    __init__.py
    tools.py
```

然后可以 `import demo_pkg`。

In [ ]:
pkg_zip = lab / "package_modules.zip"
with zipfile.ZipFile(pkg_zip, "w") as zf:
    zf.writestr("demo_pkg/__init__.py", "from .tools import describe\n")
    zf.writestr(
        "demo_pkg/tools.py",
        "def describe():\n"
        "    return 'package import from zip works'\n",
    )

for name in ["demo_pkg", "demo_pkg.tools"]:
    sys.modules.pop(name, None)

if str(pkg_zip) not in sys.path:
    sys.path.insert(0, str(pkg_zip))
importlib.invalidate_caches()

import demo_pkg

print("demo_pkg.__file__:", demo_pkg.__file__)
print("demo_pkg.describe():", demo_pkg.describe())
assert demo_pkg.describe() == "package import from zip works"

## 5. 看看 Python 用了什么 importer

当 `sys.path` 里有 zip 文件时，Python 会为这个路径创建 `zipimport.zipimporter`。

In [ ]:
importer = sys.path_importer_cache.get(str(pkg_zip))
print("importer:", importer)
print("type:", type(importer))
assert isinstance(importer, zipimport.zipimporter)

## 6. 互动练习：向已有 zip 添加一个新模块

下面向 `flat_modules.zip` 追加 `gamma.py`。因为 Python 可能缓存了 zip 目录，我们会清理对应的 importer cache。

In [ ]:
with zipfile.ZipFile(flat_zip, "a") as zf:
    zf.writestr(
        "gamma.py",
        "def square(x):\n"
        "    return x * x\n",
    )

sys.modules.pop("gamma", None)
sys.path_importer_cache.pop(str(flat_zip), None)
importlib.invalidate_caches()

import gamma

print("gamma.__file__:", gamma.__file__)
print("gamma.square(6):", gamma.square(6))
assert gamma.square(6) == 36

## 7. 常见误区

- `.zip` 能 import，不代表 Python 会自动搜索任何 zip。zip 文件仍然必须在 `sys.path` 中。
- flat zip 的模块要放在 zip 根目录，才能 `import module_name`。
- package zip 需要保留目录结构，通常要有 `package/__init__.py`。
- zipimport 适合纯 Python `.py` 模块；不要指望它直接加载 native extension，比如 `.so` 文件。
- 从项目根目录运行脚本时，根目录会自动进入 `sys.path`，这是最简单的实验方式。

## 8. 和之前实验文件的关系

之前提到 zipimport 时，意思是：如果一个 zip 文件内部是这样的 flat 结构：

```text
paper2601_splitmae_client.py
paper2601_splitmae_server.py
paper2601_splitmae_utils.py
```

并且这个 zip 文件路径出现在 `sys.path` 中，那么 Python 可以执行：

```python
from paper2601_splitmae_client import Paper2601SplitMAEClient
```

但当前项目已经不推荐 zip 方式。推荐方式是直接在 Linux 主机上：

```bash
cd /path/to/dysphonia-detection
uv run --no-sync python paper2601_splitmae_cli.py smoke
```

这样项目根目录自动进入 `sys.path`，不需要额外设置。

## 9. 清理本 notebook 对当前 Python 进程的 import path 修改

下面的 cell 会从 `sys.path` 中移除本教程加入的临时目录和 zip。临时文件仍然留在 `/tmp` 下，方便你回头查看。

In [ ]:
for name in ["hello_module", "alpha", "beta", "gamma", "demo_pkg", "demo_pkg.tools"]:
    sys.modules.pop(name, None)

for path in [str(normal_dir), str(flat_zip), str(pkg_zip)]:
    while path in sys.path:
        sys.path.remove(path)
    sys.path_importer_cache.pop(path, None)

importlib.invalidate_caches()
print("removed tutorial paths from sys.path")
print("lab files remain here:", lab)